# CSE548 Advanced Network Security - Project 2 - SDN-Based Stateless Firewall

Created with Grok Build

The filter in this lab does not live in the forwarding path as a list of iptables rules. An Open vSwitch asks a controller what to do, and the controller answers with OpenFlow. POX is that controller. Mininet supplies four hosts on the one switch. A firewall application loaded into POX reads two config files and installs drop rules for the flows the lab wants blocked.


## Project overview

Software-defined networking splits the work. The switch forwards frames. The controller decides which frames are allowed to become flows. Here the switch is Open vSwitch, the southbound protocol is OpenFlow, and the controller is POX.

Mininet builds the topology as four containers on a single switch. POX is started with the stock layer-2 learning module and with a firewall module. The learning module keeps ordinary traffic moving. The firewall module reads a layer-3 config and a layer-2 config and installs the blocking flows.

The rules that ship with the lab do not separate these four hosts from each other. The hosts were renumbered, matching rules were added, and the tests below check those added rules.


## Network setup

Four hosts hang off one switch, and the switch has a single controller. Mininet's `single,4` topology is that picture. The `--mac` flag assigns each host a fixed address, `00:00:00:00:00:01` through `00:00:00:00:00:04`, in host order. Those addresses are what the layer-2 rule can name.

:::{figure} images/topology.png
One switch, four hosts, and a controller. The drawing is the lab topology with the host addresses changed.
:::

Mininet's default addresses on this topology are `10.0.0.1` through `10.0.0.4`. Each host was moved onto 192.168.2.0/24:

| Host | Address | MAC from `--mac` |
|---|---|---|
| h1 | 192.168.2.10 | 00:00:00:00:00:01 |
| h2 | 192.168.2.20 | 00:00:00:00:00:02 |
| h3 | 192.168.2.30 | 00:00:00:00:00:03 |
| h4 | 192.168.2.40 | 00:00:00:00:00:04 |

The first three windows already show the new address. The fourth still shows 10.0.0.4, and the next command in that window sets `h4-eth0` to 192.168.2.40.

:::{figure} images/host1-address.png
h1 is 192.168.2.10 on `h1-eth0`.
:::

:::{figure} images/host2-address.png
h2 is 192.168.2.20 on `h2-eth0`.
:::

:::{figure} images/host3-address.png
h3 is 192.168.2.30 on `h3-eth0`.
:::

:::{figure} images/host4-address.png
h4 still shows the Mininet default, 10.0.0.4, above the command that sets 192.168.2.40.
:::


## Software

POX is the controller. Mininet creates the hosts, the switch, and the links, and it points the switch at a remote controller instead of using its own. The switch image is Open vSwitch, selected with `--switch=ovsk`.

The controller process is:

```text
./pox.py openflow.of_01 --port=6655 pox.forwarding.l2_learning pox.forwarding.L3Firewall --l2config="l2firewall.config" --l3config="l3firewall.config"
```

`openflow.of_01 --port=6655` is the OpenFlow listener. `l2_learning` installs ordinary forwarding flows so the switch does not have to ask about every frame. `L3Firewall` reads the two config files and installs the drop rules. The layer-2 file is not a separate process. It is an argument to the same firewall component.

The network is:

```text
sudo mn --topo=single,4 --controller=remote,port=6655 --switch=ovsk --mac
```

That starts h1 through h4, switch s1, and a link from each host to the switch, with the controller expected at port 6655.


## Project description

Each host address was set from a shell on that host with `ifconfig`, as the four windows above show. The firewall files were then extended. Rules 1 through 6 in the layer-3 file are the lab's original rows, written against the 10.0.0.0 addresses Mininet uses before the renumber. Rules 7 through 11 are the rows added for 192.168.2.0/24.

:::{figure} images/l3-rules.png
Layer-3 rules. Rows 7 through 11 are the ones added for the renumbered hosts.
:::

Each row is a priority, a source and destination MAC, a source and destination IP, a source and destination port, and a protocol. `any` leaves that field unmatched. The added rows are:

- Rule 7 drops ICMP from 192.168.2.10 to 192.168.2.30.
- Rule 8 drops ICMP from 192.168.2.20 to 192.168.2.40.
- Rule 9 drops TCP port 80 to 192.168.2.20 from any source.
- Rule 10 drops TCP from 192.168.2.10 to 192.168.2.20 on any port.
- Rule 11 drops UDP from 192.168.2.10 to 192.168.2.20.

:::{figure} images/l2-rules.png
Layer-2 rules. Rule 3 drops frames from `00:00:00:00:00:02` to `00:00:00:00:00:04`, which is h2 toward h4.
:::

The layer-2 file matches MAC addresses only. Rules 1 and 2 were already there. Rule 3 is the one added for this topology, and it names h2 and h4 because `--mac` numbered them that way.

:::{figure} images/switch-config.png
Bridge s1 is connected to the controller at 127.0.0.1:6655. Open vSwitch here is 2.9.8.
:::

`ovs-vsctl show` is the check that the switch really joined the controller. Bridge s1 has a port for each host, `is_connected` is true, and the controller address is `tcp:127.0.0.1:6655`, the port POX was given. The bridge also listens on `ptcp:6654`, which is Open vSwitch's own passive OpenFlow port. `fail_mode` is `secure`: if the switch loses the controller, it drops traffic instead of falling back to normal forwarding.

### ICMP from h1 to h3

:::{figure} images/ping-h1-h3.png
One echo reply comes back from h3, and the rest of the pings are lost.
:::

Rule 7 is the drop for this pair. The first reply still arrives. On a reactive controller the switch does not yet have a flow for that pair when the first packet shows up, so the packet is sent to the controller. The controller can forward that packet and only then install the drop. Later pings match the installed flow and stop. Sixty-three probes produced one reply.

### ICMP from h2 to h4

:::{figure} images/ping-h2-h4.png
h2 receives nothing from h4.
:::

Rule 8 covers this pair. Unlike the previous test, no reply got through before the drop took effect.

### HTTP to h2

:::{figure} images/http-server.png
h2 serves HTTP on port 80.
:::

:::{figure} images/http-clients.png
h1, h3, and h4 each start a request to h2. None of them receive a page.
:::

Rule 9 drops TCP port 80 to 192.168.2.20 from any source, so the server can be running and still be unreachable. The three clients hang until the request is cancelled.

### MAC filter from h2 to h4

:::{figure} images/arping.png
h2 gets a single ARP reply from h4. h1 keeps receiving replies from h3.
:::

`arping` from h2 to h4 gets one response and then nothing, which is the same first-packet pattern as the ICMP test. `arping` from h1 to h3 keeps getting replies. Rule 3 names h2's MAC and h4's MAC. It does not name h1 or h3, so that second pair is left to layer-2 learning.

### TCP and UDP from h1 to h2

:::{figure} images/tcp-hping.png
TCP probes from h1 to h2 are all lost.
:::

:::{figure} images/udp-hping.png
UDP probes from h1 to h2 are all lost.
:::

`hping3` sends the probes without depending on a listening server. Rule 10 drops the TCP, on any port. Rule 11 drops the UDP. Both tests show total loss, so those two rows are installed and matching.


## Conclusion

The switch forwarded, and the controller held the policy. Four hosts, one bridge, and two short config files were enough to separate the pairs the tests care about: ICMP between two of them, HTTP onto h2, frames from h2's MAC to h4's MAC, and TCP or UDP from h1 to h2.

The firewall also stopped answering after it had been up for a while. Starting the POX process again made the same rules work. That fits a controller that has stopped installing or refreshing flows. The switch screenshot shows `fail_mode: secure`, so a controller that is actually gone should cause the bridge to drop packets rather than forward them. A restart of POX putting the policy back says the controller process was the piece that had stalled.
